# Optimized MetroBot Fine-Tuning Pipeline
This notebook installs Unsloth dependencies cleanly, fixes the undefined path error, and maximizes Colab GPU performance.

In [ ]:
# 0. Install Optimized Dependencies for Google Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft transformers accelerate diffusers

In [ ]:
import torch
import os
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Base Configuration
max_seq_length = 2048
dtype = None  # None auto-detects (Float16 for T4, Bfloat16 for Ampere+)
load_in_4bit = True

# Path Configuration
TRAIN_PATH = "dataset/metrostack_train.jsonl"

# Graceful directory & dummy file creation to prevent immediate initialization crashes
if not os.path.exists(TRAIN_PATH):
    os.makedirs(os.path.dirname(TRAIN_PATH), exist_ok=True)
    print(f"[WARNING] '{TRAIN_PATH}' not found. Please upload your training data to the sidebar folder structure.")
    # Creating a placeholder structure so the code syntax can be verified without breaking
    with open(TRAIN_PATH, "w") as f:
        f.write('{"messages": [{"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello"}]}\n')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Setup LoRA Target Parameters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Optimized memory footprint
    random_state = 3407,
    use_rslora = False,
)

# 3. Format & Apply ChatML Templates
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files=TRAIN_PATH, split="train")

# Optimization: Utilizing multi-processing based on environment core availability
cpu_cores = os.cpu_count() or 2
dataset = dataset.map(formatting_prompts_func, batched = True, num_proc = cpu_cores)

# 4. Define High-Throughput Fine-Tuning Hyperparameters
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = cpu_cores,
    packing = False, # Set to True if your individual samples are very short to speed up processing further
    args = TrainingArguments(
        per_device_train_batch_size = 4,      # Increased from 2 to maximize VRAM utilization for 1.5B model
        gradient_accumulation_steps = 2,      # Adjusted to keep effective global batch size optimal
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",                 # Keeps memory overhead low
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 5. Run Training Loop
print("--- STARTING METROBOT TRAINING LOOP ---")
trainer_stats = trainer.train()

# 6. Direct GGUF Extraction
print("--- PACKAGING COMPLETED MODEL TO GGUF ---")
model.save_pretrained_gguf("metrobot_gguf", tokenizer, quantization_method = "q4_k_m")
print("--- PIPELINE SUCCESSFUL. YOU CAN NOW DOWNLOAD THE GGUF FILE FROM THE SIDEBAR ---")